In [21]:
import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from collections import Counter
import torch.optim as optim




In [5]:
df = pd.read_csv("reddit_depression_dataset.csv")


C:\Users\11\AppData\Local\Temp\ipykernel_3536\3194037386.py:1: DtypeWarning: Columns (0: Unnamed: 0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("reddit_depression_dataset.csv")


In [6]:
df.columns

Index(['Unnamed: 0', 'subreddit', 'title', 'body', 'upvotes', 'created_utc',
       'num_comments', 'label'],
      dtype='str')

In [7]:

df = df.drop(columns=['Unnamed: 0'])

In [8]:

df['full_text'] = df['title'] + " " + df['body']

In [9]:
print(df.isnull().sum())

df['full_text'] = df['full_text'].fillna('')

subreddit           20
title               23
body            461051
upvotes             64
created_utc        106
num_comments    113977
label              106
full_text       461053
dtype: int64


In [10]:
df = df.dropna(subset=['label'])

In [11]:

df['title'] = df['title'].fillna('')
df['body'] = df['body'].fillna('')


df['full_text'] = df['title'] + " " + df['body']

df = df[df['full_text'].str.strip() != ""]

In [12]:
df = df[['full_text', 'label']]

In [13]:


def clean_text(text):
    text = text.lower() # تحويل لصغير
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) 
    text = text.translate(str.maketrans('', '', string.punctuation))
   
    return text

df['full_text'] = df['full_text'].apply(clean_text)

In [14]:
df.isnull().sum()

full_text    0
label        0
dtype: int64

In [15]:
print(df['label'].value_counts())

label
0.0    1990260
1.0     480411
Name: count, dtype: int64


In [16]:


df_majority = df[df['label'] == 0.0]
df_minority = df[df['label'] == 1.0]


df_majority_downsampled = df_majority.sample(n=len(df_minority), random_state=42)


df_balanced = pd.concat([df_majority_downsampled, df_minority])


df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)


print(df_balanced['label'].value_counts())

label
0.0    480411
1.0    480411
Name: count, dtype: int64


In [17]:
X = df_balanced['full_text']
y = df_balanced['label']

In [ ]:
class MentalHealthLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super(MentalHealthLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        out = self.fc(hidden[-1])
        return self.sigmoid(out)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


model = MentalHealthLSTM(vocab_size=len(vocab), embed_dim=64, hidden_dim=128, output_dim=1).to(device)



✅ Model Initialized on cpu
حجم القاموس المظبوط: 10002


In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:

all_words = ' '.join(X_train).split()
word_counts = Counter(all_words)
vocab = {word: i+2 for i, (word, count) in enumerate(word_counts.most_common(10000))}
vocab['<PAD>'] = 0  
vocab['<UNK>'] = 1  


def tokenize_text(text, max_len=100):
    tokens = [vocab.get(word, 1) for word in str(text).split()]
   
    if len(tokens) < max_len:
        tokens += [0] * (max_len - len(tokens))
    else:
        tokens = tokens[:max_len]
    return np.array(tokens)


X_train_seq = np.array([tokenize_text(t) for t in X_train])
X_test_seq = np.array([tokenize_text(t) for t in X_test])


y_train_tensor = torch.FloatTensor(y_train.values)
y_test_tensor = torch.FloatTensor(y_test.values)


train_data = TensorDataset(torch.LongTensor(X_train_seq), y_train_tensor)
test_data = TensorDataset(torch.LongTensor(X_test_seq), y_test_tensor)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)



⏳ جاري تحويل النصوص لأرقام... لحظات
✅ تم تجهيز train_loader و test_loader بنجاح!
حجم القاموس: 10002 كلمة


C:\Users\11\AppData\Local\Temp\ipykernel_3536\133849226.py:25: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  y_train_tensor = torch.FloatTensor(y_train.values)


In [ ]:

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 3
model.train()

print(f"🚀 Training Started...")
for epoch in range(epochs):
    total_loss = 0
   
    for batch_idx, (texts, labels) in enumerate(train_loader):
        texts, labels = texts.to(device), labels.to(device)
        
       
        optimizer.zero_grad()
        
      
        outputs = model(texts).squeeze() 
        
      
        loss = criterion(outputs, labels.float()) 
        
     
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
      
        if (batch_idx + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx+1}/{len(train_loader)}] | Current Loss: {loss.item():.4f}")
            

    print(f"✅ End of Epoch [{epoch+1}/{epochs}] | Average Loss: {total_loss/len(train_loader):.4f}")
    print("-" * 50)

🚀 Training Started...
Epoch [1/3] | Batch [50/24021] | Current Loss: 0.2372
Epoch [1/3] | Batch [100/24021] | Current Loss: 0.2647
Epoch [1/3] | Batch [150/24021] | Current Loss: 0.2237
Epoch [1/3] | Batch [200/24021] | Current Loss: 0.2179
Epoch [1/3] | Batch [250/24021] | Current Loss: 0.3490
Epoch [1/3] | Batch [300/24021] | Current Loss: 0.1471
Epoch [1/3] | Batch [350/24021] | Current Loss: 0.2077
Epoch [1/3] | Batch [400/24021] | Current Loss: 0.0809
Epoch [1/3] | Batch [450/24021] | Current Loss: 0.2956
Epoch [1/3] | Batch [500/24021] | Current Loss: 0.0831
Epoch [1/3] | Batch [550/24021] | Current Loss: 0.3050
Epoch [1/3] | Batch [600/24021] | Current Loss: 0.2711
Epoch [1/3] | Batch [650/24021] | Current Loss: 0.2149
Epoch [1/3] | Batch [700/24021] | Current Loss: 0.1679
Epoch [1/3] | Batch [750/24021] | Current Loss: 0.2525
Epoch [1/3] | Batch [800/24021] | Current Loss: 0.2438
Epoch [1/3] | Batch [850/24021] | Current Loss: 0.2943
Epoch [1/3] | Batch [900/24021] | Current Lo